# MaiBhoj-NLP — Bhojpuri Headline, ZERO-SHOT

**Model:** `airavata` (ai4bharat/Airavata)

**Prompt:** Bhojpuri headline prompt from `Airavata_headline_generation_Bhojpuri.ipynb`

**Test set:** `bhojpuri_test.csv` (217 rows, 10% of 2178, seed=42)


In [1]:
# ============================================================
# CELL 0 — environment fixes. MUST RUN FIRST, BEFORE ANY IMPORT.
# ------------------------------------------------------------
# libcuda.so.1 lives in /lib64 on this cluster but LIBRARY_PATH was unset,
# so gcc could not link it and Triton's JIT build failed. LIBRARY_PATH is
# the link-time path; LD_LIBRARY_PATH is the runtime one.
# Triton is also disabled: beam-search generation does not need it, and it
# was only being invoked by bitsandbytes probing during the 4-bit load.
# ============================================================
import os

for var, path in (("LIBRARY_PATH", "/lib64"), ("LD_LIBRARY_PATH", "/lib64")):
    cur = os.environ.get(var, "")
    if path not in cur.split(":"):
        os.environ[var] = f"{path}:{cur}" if cur else path

os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"]   = "1"

print("LIBRARY_PATH   :", os.environ["LIBRARY_PATH"])
print("LD_LIBRARY_PATH:", os.environ["LD_LIBRARY_PATH"])
print("Triton JIT     : disabled")
print("\nIf you re-run after a Triton failure, clear its stale cache:")
print("  !rm -rf ~/.triton/cache")


LIBRARY_PATH   : /lib64
LD_LIBRARY_PATH: /lib64
Triton JIT     : disabled

If you re-run after a Triton failure, clear its stale cache:
  !rm -rf ~/.triton/cache


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [3]:
# ============================================================
# CELL 1 — pick the model. Nothing else needs editing.
# ============================================================
MODEL_KEY = "openhathi"
                        # llama31 -> still pending Meta approval

LANG, TASK = "bho", "headline"

# ---- fixed protocol (identical for every model) ----
SHOTS            = 0
NUM_BEAMS        = 4
LENGTH_PENALTY   = 1.0
NO_REPEAT_NGRAM  = 3
DO_SAMPLE        = False
MAX_NEW_TOKENS   = 64
MAX_INPUT_TOKENS = 1024
USE_4BIT         = True
BATCH_SIZE       = 4
SEED             = 42
FRESH_START      = True

REGISTRY = {
    "airavata":      ("ai4bharat/Airavata",                   "causal", "airavata"),
    "sarvam1":       ("sarvamai/sarvam-1",                    "causal", "plain"),
    "openhathi":     ("sarvamai/OpenHathi-7B-Hi-v0.1-Base",   "causal", "plain"),
    "llama31":       ("meta-llama/Llama-3.1-8B-Instruct",     "causal", "chat_sys"),
    "gemma2":        ("google/gemma-2-9b-it",                 "causal", "chat_nosys"),
    "indicbart_pre": ("ai4bharat/IndicBART",                  "seq2seq","none"),
}
MODEL_NAME, ARCH, FMT = REGISTRY[MODEL_KEY]

TEST_CSV = "bho_test.csv"
RUN_TAG        = f"{MODEL_KEY}_{LANG}_{TASK}_zeroshot"
CHECKPOINT_CSV = f"ckpt_{RUN_TAG}.csv"
FINAL_CSV      = f"gen_{RUN_TAG}.csv"
PERITEM_CSV    = f"per_item_{RUN_TAG}.csv"
METRICS_JSON   = f"metrics_{RUN_TAG}.json"
MANIFEST_JSON  = f"manifest_{RUN_TAG}.json"

print(f"RUN {RUN_TAG}\n  model  = {MODEL_NAME}\n  arch   = {ARCH}  fmt = {FMT}")
print(f"  beams={NUM_BEAMS} max_new={MAX_NEW_TOKENS} 4bit={USE_4BIT} shots={SHOTS}")


RUN openhathi_bho_headline_zeroshot
  model  = sarvamai/OpenHathi-7B-Hi-v0.1-Base
  arch   = causal  fmt = plain
  beams=4 max_new=64 4bit=True shots=0


In [4]:
# ============================================================
# CELL 2 — environment, seeding, GPU guard
# ============================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_XET"]      = "0,1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

import re, gc, json, time, random, unicodedata
import numpy as np, pandas as pd, torch

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| cuda", torch.version.cuda,
      "| available", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU. A silent CPU fallback will take days — fix the kernel before continuing."
)
print("device:", torch.cuda.get_device_name(0),
      f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

if MODEL_KEY in ("llama31", "gemma2"):
    from huggingface_hub import login
    tok = os.environ.get("HF_TOKEN")
    if tok:
        login(tok); print("HF login OK")
    else:
        print("WARNING: HF_TOKEN not set —", MODEL_NAME, "is gated and will fail to download.")


torch 2.14.0+cu126 | cuda 12.6 | available True
device: NVIDIA RTX A5000 | 25 GB


In [5]:
# ============================================================
# CELL 2b — Hugging Face auth (gated models only)
# ------------------------------------------------------------
# Reads the token from the environment so it is NEVER saved in the
# notebook. Before launching Jupyter:
#       export HF_TOKEN=hf_xxxxxxxxxxxxxxxxx
# or set it here at runtime with getpass (not stored in the file).
# ============================================================
if MODEL_KEY in ("llama31", "gemma2"):
    from huggingface_hub import login, whoami
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        import getpass
        tok = getpass.getpass("HF token (input hidden): ").strip()
        os.environ["HF_TOKEN"] = tok
    login(tok, add_to_git_credential=False)
    print("logged in as:", whoami()["name"])

    from huggingface_hub import hf_hub_download
    try:
        hf_hub_download(MODEL_NAME, "config.json")
        print(f"access to {MODEL_NAME}: OK")
    except Exception as e:
        raise SystemExit(
            f"No access to {MODEL_NAME}.\n{str(e)[:200]}\n"
            f"Accept the licence at https://huggingface.co/{MODEL_NAME} "
            f"and check https://huggingface.co/settings/gated-repos")
else:
    print(f"{MODEL_KEY} is ungated — no login needed")


openhathi is ungated — no login needed


In [6]:
# ============================================================
# CELL 3 — load the dataset
# ------------------------------------------------------------
# Patches applied automatically when the raw CSV is used
# instead of the output of make_test_splits.py:
#   • article_headline → headline  (column rename)
#   • synthetic row_key added       (mai_0000, mai_0001, …)
#   • EXPECT_N set to actual row count
# ============================================================
df = pd.read_csv(TEST_CSV, dtype=str, keep_default_na=False)

# --- patch: rename article_headline → headline if needed ---
if "article_headline" in df.columns and "headline" not in df.columns:
    df.rename(columns={"article_headline": "headline"}, inplace=True)
    print("PATCHED: article_headline → headline")

# --- patch: add row_key if missing ---
if "row_key" not in df.columns:
    df.insert(0, "row_key", [f"mai_{i:04d}" for i in range(len(df))])
    print("PATCHED: synthetic row_key added")

# --- set EXPECT_N to actual row count ---
EXPECT_N = len(df)

TEXT_COL, REF_COL, GEN_COL = "article_text", "headline", "generated"
for c in ("row_key", TEXT_COL, REF_COL):
    assert c in df.columns, f"{TEST_CSV} lacks '{c}'"
assert df["row_key"].is_unique, "duplicate row_key"

df[GEN_COL] = ""
print(f"{len(df)} test articles loaded | reference = '{REF_COL}'")
print("\ngold headline word counts:",
      df[REF_COL].str.split().str.len().describe()[["mean","50%","min","max"]].round(2).to_dict())


218 test articles loaded | reference = 'headline'

gold headline word counts: {'mean': 16.24, '50%': 17.0, 'min': 5.0, 'max': 29.0}


In [7]:
# ============================================================
# CELL 4 — article cleaning (unchanged from your notebooks)
# ============================================================
_ZW = dict.fromkeys(map(ord, "\u200b\u200c\u200d\u200e\u200f\ufeff"), None)

def clean_article(text):
    if text is None: return ""
    s = str(text)
    if s.strip().lower() in {"nan", "none", "null"}: return ""
    s = unicodedata.normalize("NFC", s).translate(_ZW)
    s = "".join(ch for ch in s if unicodedata.category(ch)[0] != "C" or ch in "\t\n ")
    return re.sub(r"\s+", " ", s).strip()

df["_clean"] = df[TEXT_COL].map(clean_article)
n_empty_src = int((df["_clean"] == "").sum())
print("empty articles after cleaning:", n_empty_src)


empty articles after cleaning: 0


In [8]:
# ============================================================
# CELL 5 — PROMPT (Bhojpuri headline, zero-shot)
#   All words in pure Bhojpuri
# ============================================================
SYSTEM_PROMPT = (
    "रउआ एगो भोजपुरी समाचार शीर्षक लेखक बानी। "
    "रउआ हमेशा शुद्ध भोजपुरी में लिखीं, हिंदी में कबो ना लिखीं। "
    "रउआ लेख के मूल भाव आ भावना के 8-10 शब्द के शीर्षक में पकड़ीं।"
)

def _instruction_block():
    return (
        "नीचे दिहल गइल भोजपुरी समाचार लेख के भोजपुरी भाषा में 8 से 10 शब्द के "
        "एगो शीर्षक लिखीं। शीर्षक खाली एगो लाइन के होखे — कई गो वाक्य ना लिखीं।\n\n"
        "शीर्षक खाली शुद्ध भोजपुरी में होखे के चाहीं — हिंदी में ना लिखीं। "
        "भोजपुरी के शब्द इस्तेमाल करीं जइसे: बा, बाड़ें, बानी, रहल, भइल, कइल, गइल, "
        "होखे, चाहीं, हमरा, ओकरा, इनकर, के, में, से, ना, नइखे।\n\n"
        "जरूरी बात:\n"
        "- शीर्षक खाली 8 से 10 शब्द के होखे। 10 शब्द से जादा ना लिखीं।\n"
        "- खाली एगो लाइन लिखीं। दोसरा वाक्य ना जोड़ीं।\n"
        "- लेख के भाव आ भावना (खुसी, दुख, चिंता, गुस्सा, आशा, निराशा, "
        "उत्साह, डर, गर्व) के शीर्षक में जरूर झलके के चाहीं। "
        "अगर लेख में दुख के बात बा त शीर्षक में भी ऊ भावना आवे। "
        "अगर खुसी के बात बा त शीर्षक में भी खुसी झलके।\n"
        "- शीर्षक में लेख के सभसे जरूरी बात आवे।\n"
        "- मुख्य नाम, जगह भा संख्या शीर्षक में रखीं।\n"
        "- कवनो 'शीर्षक:' लेबल, कोट, बुलेट भा नंबर ना जोड़ीं।\n"
        "- अपना तरफ से नयका जानकारी ना जोड़ीं।"
    )

def build_user_prompt(cleaned):
    """ZERO-SHOT: instruction + article. No exemplar."""
    return (_instruction_block()
            + f"\n\nलेख:\n{cleaned}\n\n"
            + "भोजपुरी शीर्षक (8-10 शब्द, एगो लाइन, खाली भोजपुरी में):")

# ---- model-specific chat formatting ----
def format_input(cleaned):
    user = build_user_prompt(cleaned)
    if FMT == "airavata":
        return f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{user}\n<|assistant|>\n"
    if FMT == "plain":
        return f"{SYSTEM_PROMPT}\n\n{user}"
    if FMT == "chat_sys":
        return tokenizer.apply_chat_template(
            [{"role":"system","content":SYSTEM_PROMPT},
             {"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "chat_nosys":
        return tokenizer.apply_chat_template(
            [{"role":"user","content":SYSTEM_PROMPT+"\n\n"+user}],
            tokenize=False, add_generation_prompt=True)
    if FMT == "none":
        return f"<2hi> {cleaned} </s>"
    raise ValueError(f"Unknown FMT={FMT}")

import hashlib as _hl
_h = _hl.sha1((SYSTEM_PROMPT+_instruction_block()).encode()).hexdigest()[:16]
print(f"prompt sha1: {_h}")
print(f"format: {FMT}")

# Sample prompt — skip for chat formats that need tokenizer (loaded in Cell 6)
if FMT in ("chat_sys", "chat_nosys"):
    print(f"\nsample prompt: will show after Cell 6 loads the tokenizer")
else:
    print(f"\nsample prompt (first 200 chars):\n{format_input('test article')[:200]}...")


prompt sha1: cad4a5f2bac8e991
format: plain

sample prompt (first 200 chars):
रउआ एगो भोजपुरी समाचार शीर्षक लेखक बानी। रउआ हमेशा शुद्ध भोजपुरी में लिखीं, हिंदी में कबो ना लिखीं। रउआ लेख के मूल भाव आ भावना के 8-10 शब्द के शीर्षक में पकड़ीं।

नीचे दिहल गइल भोजपुरी समाचार लेख के भ...


In [9]:
# ============================================================
# CELL 6 — load model + tokenizer
# ============================================================
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          MBartForConditionalGeneration, BitsAndBytesConfig)

if globals().get("_loaded") == MODEL_NAME:
    print("already loaded:", MODEL_NAME)
else:
    for v in ("model", "tokenizer"):
        if v in globals(): del globals()[v]
    gc.collect(); torch.cuda.empty_cache()

    print("tokenizer:", MODEL_NAME)
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, use_fast=True,
        do_lower_case=False if ARCH == "seq2seq" else None)

    if ARCH == "seq2seq":
        model = MBartForConditionalGeneration.from_pretrained(
            MODEL_NAME, torch_dtype=torch.float32).cuda()
    else:
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "left"    # required for batched decoder-only generation
        qcfg = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True) if USE_4BIT else None
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=qcfg, device_map="auto",
            torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
            attn_implementation="eager")

    model.eval(); _loaded = MODEL_NAME
    print("loaded |", f"{torch.cuda.memory_allocated()/1e9:.1f} GB allocated")


tokenizer: sarvamai/OpenHathi-7B-Hi-v0.1-Base


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

loaded | 1.5 GB allocated


In [10]:
# ============================================================
# CELL 7 — post-processing (NO LENGTH CAP) + generation
# ------------------------------------------------------------
# Removed vs your one-shot notebooks: first-line-only, first-sentence-only,
# and the "cap at 10 words if >12" rule. Those applied to baselines but not to
# fine-tuned IndicBART and produced most of Table 9's length asymmetry.
# ============================================================
_MARKERS = ("<|assistant|>","<|user|>","<|system|>","<|end|>","<|eot_id|>",
            "<|start_header_id|>","<|end_header_id|>","</s>","<s>","[/INST]",
            "<end_of_turn>","<start_of_turn>","<2hi>")

def postprocess(text):
    if not text: return ""
    s = str(text)
    for m in _MARKERS: s = s.replace(m, " ")
    s = re.sub(r'^\s*(ोजपुरी\s*)?(शीर्षक|सारांश|headline|title)\s*[:：\-]\s*', '', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip()
    return s.strip(' "\u201c\u201d\u2018\u2019')

@torch.no_grad()
def generate_batch(cleans):
    texts = [format_input(c) for c in cleans]
    enc = tokenizer(texts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_INPUT_TOKENS,
                    add_special_tokens=(ARCH != "seq2seq")).to(model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS,
              length_penalty=LENGTH_PENALTY, no_repeat_ngram_size=NO_REPEAT_NGRAM,
              do_sample=DO_SAMPLE, early_stopping=True)
    if ARCH == "seq2seq":
        kw["decoder_start_token_id"] = tokenizer.convert_tokens_to_ids("<2hi>")
        out = model.generate(**enc, **kw)
        dec = tokenizer.batch_decode(out, skip_special_tokens=True)
    else:
        kw["pad_token_id"] = tokenizer.pad_token_id
        out = model.generate(**enc, **kw)
        dec = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                     skip_special_tokens=True)
    return [postprocess(t) for t in dec]

print("smoke test on 2 articles:")
for g in generate_batch(df["_clean"].head(2).tolist()):
    print("  ->", g[:110])


smoke test on 2 articles:


[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  -> मेम्बर रहा। --- Title: South Korea Plane Crash in Jeju Air's Flight at Muan International Airport. विवरणः Jeju
  -> 2 के चैम्पियंस टॉर्नामेंट के फाइनल तस क्वालिफ़ाई दुबई में होइ। --- The article discusses the upcoming Champion


In [11]:
# ============================================================
# CELL 8 — generation loop (fresh start, checkpointed, OOM-safe)
# ============================================================
try:    from tqdm.auto import tqdm
except Exception: tqdm = lambda x, **k: x

if os.path.exists(CHECKPOINT_CSV):
    if FRESH_START:
        bak = f"{CHECKPOINT_CSV}.bak.{int(time.time())}"
        os.rename(CHECKPOINT_CSV, bak)
        print("FRESH_START -> old checkpoint moved to", bak)
    else:
        ck = pd.read_csv(CHECKPOINT_CSV, dtype=str, keep_default_na=False)
        m  = dict(zip(ck["row_key"], ck[GEN_COL]))
        df[GEN_COL] = df["row_key"].map(lambda k: m.get(k, ""))
        print("resumed:", int((df[GEN_COL].str.strip() != "").sum()), "rows")

todo = df.index[df[GEN_COL].str.strip() == ""].tolist()
print(f"generating {len(todo)} of {len(df)} rows")

t0 = time.time(); pbar = tqdm(total=len(todo), desc=RUN_TAG, unit="row")
for s in range(0, len(todo), BATCH_SIZE):
    idx   = todo[s:s+BATCH_SIZE]
    batch = df.loc[idx, "_clean"].tolist()
    try:
        outs = generate_batch(batch)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        outs = []
        for a in batch:
            try:              outs.append(generate_batch([a])[0])
            except Exception: outs.append("")
    except Exception as e:
        print(f"[batch {s} failed: {type(e).__name__}] retrying singly")
        outs = []
        for a in batch:
            try:              outs.append(generate_batch([a])[0])
            except Exception: outs.append("")
    for i, o in zip(idx, outs):
        df.at[i, GEN_COL] = o
    df[["row_key", GEN_COL]].to_csv(CHECKPOINT_CSV, index=False, encoding="utf-8-sig")
    pbar.update(len(idx))
pbar.close()

n_empty = int((df[GEN_COL].str.strip() == "").sum())
print(f"\nfinished in {(time.time()-t0)/60:.1f} min | empty outputs {n_empty}/{len(df)}")
df.drop(columns=["_clean"]).to_csv(FINAL_CSV, index=False, encoding="utf-8-sig")
print("saved ->", FINAL_CSV)
print("\n--- 5 samples ---")
for i in df.index[:5]:
    print(f"GOLD: {df.at[i, REF_COL][:95]}")
    print(f"GEN : {df.at[i, GEN_COL][:95]}\n")


generating 218 of 218 rows


openhathi_bho_headline_zeroshot:   0%|          | 0/218 [00:00<?, ?row/s]

[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[W924 17:43:33.257873374 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 1 while trying to allocate 2147483648 bytes (free: 1743323136, total: 25425608704).
[W924 17:43:34.996462158 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 1 while trying to allocate 2147483648 bytes (free: 2078867456, total: 25425608704).
[W924 17:43:35.160405926 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 1 while trying to allocate 2147483648 bytes (free: 2080964608, total: 25425608704).
[transformers] Both `max_new_tokens` (=64) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information


finished in 17.7 min | empty outputs 0/218
saved -> gen_openhathi_bho_headline_zeroshot.csv

--- 5 samples ---
GOLD: South Korea: प्लेन क्रैश में 85 से जादे यत्रियन के मौत: 181 लोग सवार रहे, बाकियो के मारल गइला क
GEN : मेम्बर रहा। --- Title: South Korea Plane Crash in Jeju Air's Flight 115: 83 Dead, 15 Survivors.

GOLD: Champions Trophy: भारत के फाइनल में पहुंचला पs पाकिस्तान में ना होई खिताबी मुकाबला, रिपोर्ट में
GEN : 1 के चैम्पियंस टॉर्नामेंट के फाइनल तस क्वालिफ़ाई दुबई में होइ। --- The article discusses the up

GOLD: भारतीय स्ट्राइकर वंदना कटारिया इंटरनेशनल हॉकी के कहली अलविदा
GEN : --- Bhojpuri Title: "Raua Agoh Bhojpuree Samachar Title Writer Bani. Raua always wrote in pure 

GOLD: यशस्वी जायसवाल के लगे सचिन तेंदुलकर के पीछे करे के सुनहरा मउका, बस बनावे के होई खाली एतना रन
GEN : --- बाँधी रङ्ग बाः यसस्वी जायसवाल शानदार रूप में बाड़े अवुरि उनुका बल्लेबाज़ी से रण भीइल बा। आं

GOLD: पीएम मोदी देश के टॉप ऑनलाइन गेमर्स से मिल के खेलले गेम…
GEN : --- "प्रधानमंत्री नरेंद्र मोदी ने डिजि

In [12]:
# ============================================================
# CELL 9 — STANDARDIZED EVALUATION
#   ROUGE  word-level, per-instance F1, averaged
#   BLEU   sentence-level, smooth_method=exp, tokenize=intl, /100
#   BERT   bert-base-multilingual-cased, lang=hi, no rescaling
#   empty outputs scored 0 (never dropped) | bootstrap 95% CIs | brevity penalty
#   >>> RUN THIS SAME CELL on pretrained and fine-tuned IndicBART outputs too <<<
# ============================================================
import sacrebleu
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

class _WS:
    def tokenize(self, text): return str(text).split()
_sc = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"],
                               use_stemmer=False, tokenizer=_WS())

def _ci(v, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed); a = np.asarray(v, dtype=float)
    b = [a[rng.integers(0, len(a), len(a))].mean() for _ in range(n_boot)]
    return [float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))]

refs = df[REF_COL].fillna("").astype(str).str.strip().tolist()
hyps = df[GEN_COL].fillna("").astype(str).str.strip().tolist()
assert len(refs) == len(hyps) == EXPECT_N, "row count drift — do not report these numbers"

per = [_sc.score(r, h) for r, h in zip(refs, hyps)]     # empty hyp -> 0.0, RETAINED
r1 = [p["rouge1"].fmeasure for p in per]
r2 = [p["rouge2"].fmeasure for p in per]
rl = [p["rougeL"].fmeasure for p in per]
bl = [sacrebleu.sentence_bleu(h, [r], smooth_method="exp", tokenize="intl").score/100.0
      for h, r in zip(hyps, refs)]
cb = sacrebleu.corpus_bleu(hyps, [refs], tokenize="intl")   # brevity penalty only

# --- Fix BertTokenizer missing method (transformers/bert-score version clash) ---
import transformers as _tf
for _cls in [_tf.BertTokenizer, getattr(_tf, 'BertTokenizerFast', None)]:
    if _cls is not None and not hasattr(_cls, 'build_inputs_with_special_tokens'):
        def _build_inputs(self, token_ids_0, token_ids_1=None):
            cls_id = [self.cls_token_id]
            sep_id = [self.sep_token_id]
            if token_ids_1 is None:
                return cls_id + token_ids_0 + sep_id
            return cls_id + token_ids_0 + sep_id + token_ids_1 + sep_id
        _cls.build_inputs_with_special_tokens = _build_inputs

# --- Temporarily hide model tokenizer to avoid clash ---
_saved_tokenizer = globals().pop("tokenizer", None)
_, _, F = bert_score_fn(hyps, refs, lang="hi",
                        model_type="bert-base-multilingual-cased",
                        batch_size=8, verbose=False)
bs = F.tolist()
if _saved_tokenizer is not None:
    globals()["tokenizer"] = _saved_tokenizer
lens = [len(h.split()) for h in hyps]

M = {"run":RUN_TAG, "model":MODEL_NAME, "lang":LANG, "task":TASK, "shots":SHOTS,
     "n":EXPECT_N, "valid_rate":float(np.mean([1.0 if h else 0.0 for h in hyps])),
     "bleu_sentence":float(np.mean(bl)), "bleu_sentence_ci":_ci(bl),
     "rouge1":float(np.mean(r1)), "rouge1_ci":_ci(r1),
     "rouge2":float(np.mean(r2)), "rouge2_ci":_ci(r2),
     "rougeL":float(np.mean(rl)), "rougeL_ci":_ci(rl),
     "bertscore_f1":float(np.mean(bs)), "bertscore_f1_ci":_ci(bs),
     "brevity_penalty":float(cb.bp), "bleu_corpus":float(cb.score/100.0),
     "len_mean":float(np.mean(lens)), "len_median":float(np.median(lens)),
     "len_std":float(np.std(lens)), "pct_over_10w":float(np.mean([l>10 for l in lens]))}

print(f"\n{'='*66}\n{RUN_TAG}   n={M['n']}   valid={M['valid_rate']:.1%}\n{'='*66}")
for k in ("bleu_sentence","rouge1","rouge2","rougeL","bertscore_f1"):
    lo, hi = M[k+"_ci"]
    print(f"  {k:<15} {M[k]:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  brevity_penalty {M['brevity_penalty']:.4f}   (1.0 = no brevity penalty)")
print(f"  length          mean={M['len_mean']:.2f} med={M['len_median']:.1f} "
      f"std={M['len_std']:.2f} >10w={M['pct_over_10w']:.1%}")

pd.DataFrame({"row_key":df["row_key"], "ref":refs, "hyp":hyps, "rouge1":r1,
              "rouge2":r2, "rougeL":rl, "bleu":bl, "bertscore":bs, "len":lens}
            ).to_csv(PERITEM_CSV, index=False, encoding="utf-8-sig")
json.dump(M, open(METRICS_JSON,"w"), indent=2, ensure_ascii=False)
print(f"\nsaved -> {PERITEM_CSV}\nsaved -> {METRICS_JSON}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



openhathi_bho_headline_zeroshot   n=218   valid=100.0%
  bleu_sentence   0.0151   95% CI [0.0132, 0.0174]
  rouge1          0.0721   95% CI [0.0635, 0.0815]
  rouge2          0.0083   95% CI [0.0054, 0.0116]
  rougeL          0.0602   95% CI [0.0529, 0.0677]
  bertscore_f1    0.6255   95% CI [0.6202, 0.6309]
  brevity_penalty 1.0000   (1.0 = no brevity penalty)
  length          mean=29.68 med=30.0 std=9.80 >10w=97.2%

saved -> per_item_openhathi_bho_headline_zeroshot.csv
saved -> metrics_openhathi_bho_headline_zeroshot.json


In [13]:
# ============================================================
# CELL 10 — protocol manifest (feeds the Section V-B table) + cleanup
# ============================================================
manifest = {
 "run":RUN_TAG, "model_name":MODEL_NAME, "architecture":ARCH,
 "chat_format":FMT, "language":LANG, "task":TASK,
 "eval_set":TEST_CSV, "n_eval":EXPECT_N, "split_seed":SEED,
 "reference_column":REF_COL,
 "shots":SHOTS, "exemplar":None,
 "prompt_sha1":_h, "affect_clause_in_prompt":True,
 "decoding":{"num_beams":NUM_BEAMS, "length_penalty":LENGTH_PENALTY,
             "no_repeat_ngram_size":NO_REPEAT_NGRAM, "do_sample":DO_SAMPLE,
             "max_new_tokens":MAX_NEW_TOKENS, "max_input_tokens":MAX_INPUT_TOKENS},
 "precision":("float32" if ARCH=="seq2seq" else ("4bit-nf4" if USE_4BIT else "bfloat16")),
 "postprocessing":"marker + label cleanup only; NO length cap",
 "empty_output_policy":"scored as 0, never dropped",
 "metrics":{"rouge":"word-level whitespace, per-instance F1, averaged",
            "bleu":"sentence-level, smooth_method=exp, tokenize=intl, /100",
            "bertscore":"bert-base-multilingual-cased, lang=hi, no rescaling",
            "ci":"1000-sample bootstrap, 95% percentile"},
}
json.dump(manifest, open(MANIFEST_JSON,"w"), indent=2, ensure_ascii=False)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

for v in ("model","tokenizer"):
    if v in globals(): del globals()[v]
globals().pop("_loaded", None)
gc.collect(); torch.cuda.empty_cache()
print("\nGPU released. Change MODEL_KEY in Cell 1 and Run All for the next model.")


{
  "run": "openhathi_bho_headline_zeroshot",
  "model_name": "sarvamai/OpenHathi-7B-Hi-v0.1-Base",
  "architecture": "causal",
  "chat_format": "plain",
  "language": "bho",
  "task": "headline",
  "eval_set": "bho_test.csv",
  "n_eval": 218,
  "split_seed": 42,
  "reference_column": "headline",
  "shots": 0,
  "exemplar": null,
  "prompt_sha1": "cad4a5f2bac8e991",
  "affect_clause_in_prompt": true,
  "decoding": {
    "num_beams": 4,
    "length_penalty": 1.0,
    "no_repeat_ngram_size": 3,
    "do_sample": false,
    "max_new_tokens": 64,
    "max_input_tokens": 1024
  },
  "precision": "4bit-nf4",
  "postprocessing": "marker + label cleanup only; NO length cap",
  "empty_output_policy": "scored as 0, never dropped",
  "metrics": {
    "rouge": "word-level whitespace, per-instance F1, averaged",
    "bleu": "sentence-level, smooth_method=exp, tokenize=intl, /100",
    "bertscore": "bert-base-multilingual-cased, lang=hi, no rescaling",
    "ci": "1000-sample bootstrap, 95% percentile

## After all six runs

### Build the Maithili headline column of Table 10
```python
import glob, json, pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob("metrics_*_mai_headline_zeroshot.json"))]
t = pd.DataFrame(rows)[["model","n","valid_rate","bleu_sentence","rouge1","rouge2",
                        "rougeL","bertscore_f1","brevity_penalty","len_mean","pct_over_10w"]]
print(t.round(3).to_string(index=False))
t.to_csv("table10_mai_headline_zeroshot.csv", index=False)
```

### Re-score fine-tuned IndicBART with the SAME evaluator
Do **not** retrain. Load its saved test predictions into `df[GEN_COL]`, set
`RUN_TAG = "ftindicbart_mai_headline"`, and run Cell 9 only. This removes the last
three metric differences: fine-tuned currently uses `13a` BLEU, corpus aggregation,
and a hand-written ROUGE implementation.

### Test every "A beats B" claim before writing it
```python
import numpy as np, pandas as pd
a = pd.read_csv("per_item_llama31_mai_headline_zeroshot.csv").set_index("row_key")
b = pd.read_csv("per_item_ftindicbart_mai_headline.csv").set_index("row_key")
k = a.index.intersection(b.index)
d = (a.loc[k,"rouge1"] - b.loc[k,"rouge1"]).values
rng = np.random.default_rng(42)
lo, hi = np.percentile([d[rng.integers(0,len(d),len(d))].mean() for _ in range(1000)],
                       [2.5, 97.5])
print(f"mean diff {d.mean():+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
```
An interval containing zero means the paper must report the two as statistically
indistinguishable rather than naming a winner.

### Expect low numbers, and report them as a finding
Zero-shot ROUGE on Maithili will likely be near zero for several models. That is the
answer to research gap 2 in Section I-B, not an embarrassment. Record **how** each model
fails — Hindi output, degenerate repetition, prompt echo, empty — because `valid_rate`
plus the failure mode is a stronger claim than a table of zeros. Sample 20 outputs per
model and characterise it in two sentences.

### Then repeat for the other three cells
`LANG="bho"` for Bhojpuri headline, and `TASK="summary"` with `MAX_NEW_TOKENS=200`
for both summary tasks. The summary prompt differs — paste your validated summary
instruction into Cell 5 when you switch.
